In [4]:
!pip install kagglehub


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("drnimishadavis/twitter-us-airline-sentiment")

print("Path to dataset files:", path)

/home/rito/ju_lab/ks_lab/exp3/.vevn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /home/rito/.cache/kagglehub/datasets/drnimishadavis/twitter-us-airline-sentiment/versions/1


In [6]:
!pip install numpy pandas


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [7]:
!ls  /home/rito/.cache/kagglehub/datasets/drnimishadavis/twitter-us-airline-sentiment/versions/1

Tweets.csv


In [8]:
import pandas as pd
import numpy as np

In [9]:
data = pd.read_csv(path + "/Tweets.csv")
df = pd.DataFrame(data)

In [16]:
columns=df.columns.tolist()
print(columns)

['tweet_id', 'airline_sentiment', 'airline_sentiment_confidence', 'negativereason', 'negativereason_confidence', 'airline', 'airline_sentiment_gold', 'name', 'negativereason_gold', 'retweet_count', 'text', 'tweet_coord', 'tweet_created', 'tweet_location', 'user_timezone']


In [17]:
df.text

0                      @VirginAmerica What @dhepburn said.
1        @VirginAmerica plus you've added commercials t...
2        @VirginAmerica I didn't today... Must mean I n...
3        @VirginAmerica it's really aggressive to blast...
4        @VirginAmerica and it's a really big bad thing...
                               ...                        
14635    @AmericanAir thank you we got on a different f...
14636    @AmericanAir leaving over 20 minutes Late Flig...
14637    @AmericanAir Please bring American Airlines to...
14638    @AmericanAir you have my money, you change my ...
14639    @AmericanAir we have 8 ppl so we need 2 know h...
Name: text, Length: 14640, dtype: str

In [25]:
!pip install gensim nltk bs4

  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
Using cached beautifulsoup4-4.15.0-py3-none-any.whl (109 kB)

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
# Import various modules for string cleaning
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning
import warnings
warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
import re
from nltk.corpus import stopwords

def review_to_wordlist( review, remove_stopwords=False ):
    # Function to convert a document to a sequence of words,
    # optionally removing stop words.  Returns a list of words.
    
    # 1. Remove HTML
    review_text = BeautifulSoup(review, "html.parser").get_text()
    
    # 2. Remove non-letters
    review_text = re.sub("[^a-zA-Z]"," ", review_text)
    
    # 3. Convert words to lower case and split them
    words = review_text.lower().split()
    
    # 4. Optionally remove stop words (false by default)
    if remove_stopwords:
        stops = set(stopwords.words("english"))
        words = [w for w in words if not w in stops]
    
    # 5. Return a list of words
    return(words)

In [ ]:
# Download the punkt tokenizer for sentence splitting
# (nltk >= 3.9 uses 'punkt_tab'; the old pickled 'punkt' can no longer be loaded)
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')

In [ ]:
# Load the punkt tokenizer
from nltk.tokenize import PunktTokenizer
tokenizer = PunktTokenizer("english")

# Define a function to split a review into parsed sentences
def review_to_sentences( review, tokenizer, remove_stopwords=False ):
    # Function to split a review into parsed sentences. Returns a list of sentences, where each sentence is a list of words
    
    # 1. Use the NLTK tokenizer to split the paragraph into sentences
    raw_sentences = tokenizer.tokenize(review.strip())
    
    # 2. Loop over each sentence
    sentences = []
    for raw_sentence in raw_sentences:
        # If a sentence is empty, skip it
        if len(raw_sentence) > 0:
            # Otherwise, call review_to_wordlist to get a list of words
            sentences.append( review_to_wordlist( raw_sentence,remove_stopwords ))
    
    # Return the list of sentences (each sentence is a list of words,so this returns a list of lists
    return sentences

## Word2Vec

Split every tweet into sentences (each sentence = list of words) and train a Word2Vec model on them.
Stop words are kept, since Word2Vec relies on the surrounding context to learn good vectors.

In [ ]:
sentences = []  # Initialize an empty list of sentences

print("Parsing sentences from tweets")
for tweet in df.text:
    sentences += review_to_sentences(tweet, tokenizer)

print("Total sentences:", len(sentences))
print(sentences[0])

In [ ]:
# Import the built-in logging module and configure it so that Word2Vec creates nice output messages
import logging
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

# Set values for various parameters
num_features = 100    # Word vector dimensionality
min_word_count = 3    # Minimum word count (dataset is small, so keep this low)
num_workers = 4       # Number of threads to run in parallel
context = 5           # Context window size
downsampling = 1e-3   # Downsample setting for frequent words
num_epochs = 20       # Passes over the corpus (small corpus needs more epochs)

# Initialize and train the model (skip-gram works better than CBOW on small data)
from gensim.models import Word2Vec
print("Training Word2Vec model...")
model = Word2Vec(sentences, vector_size=num_features, min_count=min_word_count,
                 window=context, workers=num_workers, sample=downsampling,
                 sg=1, epochs=num_epochs, seed=42)

print("Vocabulary size:", len(model.wv))

# Save the model for later use. It can be loaded with Word2Vec.load()
model_name = "100features_3minwords_5context"
model.save(model_name)

### Exploring the model

In [ ]:
print(model.wv.doesnt_match("bag luggage suitcase pilot".split()))
print(model.wv.most_similar("great", topn=5))
print(model.wv.most_similar("delayed", topn=5))
print(model.wv.most_similar("flight", topn=5))

### Tweet feature vectors (average of word vectors)

In [ ]:
def makeFeatureVec(words, model, num_features):
    # Function to average all of the word vectors in a given tweet
    featureVec = np.zeros((num_features,), dtype="float32")
    nwords = 0
    index2word_set = set(model.wv.index_to_key)  # words in the model's vocabulary
    for word in words:
        if word in index2word_set:
            nwords += 1
            featureVec = np.add(featureVec, model.wv[word])
    # Divide the result by the number of words to get the average
    if nwords > 0:
        featureVec = np.divide(featureVec, nwords)
    return featureVec


def getAvgFeatureVecs(reviews, model, num_features):
    # Given a set of tweets (each one a list of words), calculate the average feature vector for each one
    reviewFeatureVecs = np.zeros((len(reviews), num_features), dtype="float32")
    for counter, review in enumerate(reviews):
        reviewFeatureVecs[counter] = makeFeatureVec(review, model, num_features)
    return reviewFeatureVecs


# Stop words removed here, they only add noise to the average
clean_tweets = [review_to_wordlist(tweet, remove_stopwords=True) for tweet in df.text]
tweetDataVecs = getAvgFeatureVecs(clean_tweets, model, num_features)
print("Tweet feature vectors shape:", tweetDataVecs.shape)

### Embedding matrix for the LSTM

Index 0 is reserved for padding, so word `i` in the Word2Vec vocabulary gets index `i + 1`.
Each tweet is turned into a padded sequence of word indices that the LSTM embedding layer can use.

In [ ]:
word_index = {word: i + 1 for i, word in enumerate(model.wv.index_to_key)}
vocab_size = len(word_index) + 1  # +1 for padding index 0

embedding_matrix = np.zeros((vocab_size, num_features), dtype="float32")
for word, i in word_index.items():
    embedding_matrix[i] = model.wv[word]
print("Embedding matrix shape:", embedding_matrix.shape)

# Convert tweets to sequences of word indices (unknown words are dropped)
tweet_words = [review_to_wordlist(tweet) for tweet in df.text]
sequences = [[word_index[w] for w in words if w in word_index] for words in tweet_words]

max_len = int(np.percentile([len(s) for s in sequences], 95))
X = np.zeros((len(sequences), max_len), dtype="int64")
for i, seq in enumerate(sequences):
    seq = seq[:max_len]
    if len(seq) > 0:
        X[i, -len(seq):] = seq  # pre-padding with 0

# Sentiment labels: negative=0, neutral=1, positive=2
label_map = {"negative": 0, "neutral": 1, "positive": 2}
y = df.airline_sentiment.map(label_map).values

print("Max sequence length:", max_len)
print("X shape:", X.shape, " y shape:", y.shape)
print(X[0], y[0])